# Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Seed
seed = 42

# sklearn config
from sklearn import set_config
set_config(
    transform_output='pandas',  # `transform` and `fit_transform` returns pd.DataFrame.
    display='text',             # `pipeline` properly prints into the jupyter output cell.
)

# Summary

## Terminologies
- $X \in \mathbb{R}^{m \times n}$
  - $m$: samples, $n$: features
- $w \in \mathbb{R}^{n}$, $b \in \mathbb{R}$
- Binary classification: $y_i \in \{-1,+1\}$
- Regression: $y_i \in \mathbb{R}$.

## Prediction
- Prediction: $f(x)=w^T\phi(x)+b$
- Margin planes: $f(x)=\pm1$
- Margin width: $\frac{2}{\lVert w\rVert_2}$

## Objective
- Loss: $\displaystyle \min_{w, b} \frac12\lVert w\rVert_2^2$
- Hard margin: $s.t.\ y_i f(x_i) \ge 1$
- Soft margin: $s.t.\ y_i f(x_i) \ge 1 - \xi_i$
  - $\xi_i$ = slack variable = penalize the margin violation.

## Kernel Trick
- $w = c_1 x_1 + c_2 x_2 + ... + c_m x_m$
  - $w$ is represented as the linear combination of $m$ samples.
- $x \rightarrow \phi(x) = (x, x^2, x^3, ...)$
  - $\phi(x)$ = polynomial features
- $f(z) = w_{\phi}^T \phi(z) + b$
  - $z$ = a new sample.
  - $w_{\phi} = \displaystyle \sum_i c_i \phi(x_i)$ = weights learned from $\phi(x_i)$.
  - $ = \displaystyle \sum_{i \in SV}c_i [\phi(x_i)^T \phi(z)] + b, \quad SV = \text{support vectors}$
- $K(x, z)=\phi(x)^T\phi(z)$
  - Kernel $K(x, z)$ decides how to calculate the inner product of $x$ and $z$.

## Kernels
| Name | Formula | Key Idea | Parameters |
|---|---|---|---|
| Linear | $K(x,z)=x^Tz$ | Uses the original input features. | `kernel="linear"` |
| Polynomial | $K(x,z)=(\gamma x^Tz+r)^d$ | Captures polynomial terms and feature interactions. | `degree`, `gamma`, `coef0` |
| RBF | $K(x,z)=\exp(-\gamma\lVert x-z\rVert_2^2)$ | Responds strongly to nearby samples. | `gamma` |
| Sigmoid | $K(x,z)=\tanh(\gamma x^Tz+r)$ | Applies an S-shaped transformation to the dot product. | `gamma`, `coef0` |

$r$ = `coef0`, $d$ = `degree`. The default kernel for `SVC()` and `SVR()` is RBF.

## Practical Notes
| Item | Reminder |
|---|---|
| Scaling | Usually use `StandardScaler` inside a `Pipeline`. |
| $C$ | Smaller = stronger regularization.<br>Larger = stronger penalty on loss. |
| RBF $\gamma$ | Larger = narrower influence range.<br>Excessively large values can overfit. |
| SVR $\epsilon$ | Larger = wider unpenalized error range.<br>Depends on the target's units. |
| Kernel selection | Compare Linear and RBF using cross-validation.<br>Poly is not a remedy for multicollinearity. |
| Multiclass | `SVC`: OvO.<br>`LinearSVC`: OvR by default. |
| Probability | `decision_function()` returns scores, not probabilities. |
| `.score()` | Classification: accuracy.<br>Regression: $R^2$. |

> Note) SVM strictly requires **feature scaling**.

> Note) `SVC(decision_function_shape="ovr")` still trains using OvO. 

# Data

In [2]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=10,
    n_features=5,
    n_informative=5,
    n_redundant=0,
    n_repeated=0,
    n_classes=2,
)

X.shape

(10, 5)

# 1. LinearSVC

- SVM with a linear kernel.
- $O(m \times n)$: much faster than `SVC(kernel='linear')`.

In [3]:
from sklearn.svm import LinearSVC, LinearSVR

model = LinearSVC(
    penalty='l2',               # {'l1', 'l2}.
    loss='squared_hinge',       # {'hinge', 'squared_hinge'}.
    C=1.0,
    multi_class='ovr',
    class_weight=None,
    random_state=seed,
)

# 2. SVC

- Time complexity: $O(m^2 \times n)$ ~ $O(m^3 \times n)$.

## 1) Basic

In [4]:
from sklearn.svm import SVC

# Class
model = SVC(
    C=1.0,                          # larger C -> lower l2 penalty -> harder, more narrow margin
    gamma='scale',                  # inverse of how much each sample affects. larger gamma -> smoother boundary.
    class_weight=None,              # {'balanced'} | dict.
    decision_function_shape='ovr',  # {'ovr', 'ovo'}, for multi-class tasks.

    cache_size=200,                 # RAM size (MB). Significantly affects the execution time.
    random_state=seed,
)

# Methods
model.fit(X, y)
model.decision_function(X)  # scores, instead of classification.

# Attributes
model.support_              # idx of support vectors.
model.support_vectors_      # value of support vectors.
model.n_support_            # num of support vectors for each class.
model.classes_
model.class_weight_

array([1., 1.])

## 2) Regularization

- Lots of noise -> decrease `SVC(C)` -> higher L2 penalty.

In [5]:
model = SVC(C=1.0)      # default.
model = SVC(C=0.5)      # stronger regularization, e.g. noise.
model = SVC(C=2.0)      # simpler, broader margin.

## 3) Decision Function

In [6]:
model = SVC()
model.fit(X, y)
model.decision_function(X)  # scores, instead of classification.

array([ 0.81425019, -0.99949789,  0.99950472, -0.99950999,  0.99989618,
       -1.00048469, -0.9994979 ,  1.09004134, -0.93001556,  0.9995896 ])

## 4) Probability Prediction

> Note) `sklearn.svm.SVC(probability=True)` is deprecated.

In [7]:
from sklearn.calibration import CalibratedClassifierCV
model = CalibratedClassifierCV(
    estimator=SVC(),
    ensemble=False,
)

model.fit(X, y)
model.predict_proba(X)

array([[0.17333212, 0.82666788],
       [0.92225093, 0.07774907],
       [0.1219194 , 0.8780806 ],
       [0.92225286, 0.07774714],
       [0.12182619, 0.87817381],
       [0.92240822, 0.07759178],
       [0.92225093, 0.07774907],
       [0.10194307, 0.89805693],
       [0.91041683, 0.08958317],
       [0.12189919, 0.87810081]])

## 5) Class Weights

In [8]:
model = SVC(
    class_weight={0: 0.1, 1: 0.9},  # class k's C -> weight[k] * C.
)
model = SVC(
    class_weight='balanced',        # weight[k] = n_samples / (n_classes * n_samples_in_class).
)

## 6) Multi-class

In [9]:
X_mul, y_mul = make_classification(
    n_samples=10,
    n_features=5,
    n_informative=5,
    n_redundant=0,
    n_repeated=0,
    n_classes=4,
)

### One-versus-rest

In [10]:
model = SVC(decision_function_shape='ovr')

model.fit(X_mul, y_mul)
model.decision_function(X_mul)      # 4, per class.

array([[ 2.00605182,  3.2498141 ,  0.80303702, -0.20339219],
       [ 0.97342915, -0.21181168,  2.08086094,  3.20049606],
       [ 3.16439293,  0.96720982,  2.10877169, -0.19138987],
       [ 0.90872206, -0.20082451,  3.19549172,  2.10725521],
       [ 3.24999318,  1.85489825,  0.83130873, -0.18210759],
       [ 1.95036853,  3.25      ,  0.8018285 , -0.19202322],
       [ 3.24539614,  2.07857271,  0.82297681, -0.22096708],
       [ 0.92820691,  1.962087  , -0.1690507 ,  3.19626406],
       [ 3.24679964,  1.92093799,  0.83592071, -0.20371751],
       [ 2.17151784,  3.19994689,  0.82692743, -0.19887162]])

### One-versus-one

In [11]:
model = SVC(decision_function_shape='ovo')

model.fit(X_mul, y_mul)
model.decision_function(X_mul)      # 4x3/2 = 6, all combinations.

array([[-0.99109679,  0.55910525,  0.45048272,  0.99999999,  0.99999999,
         0.11478124],
       [ 0.38115819, -0.04899611, -0.4187791 , -0.51892247, -0.84291464,
        -0.2476422 ],
       [ 0.3493912 ,  0.16802728,  0.45566391, -0.10964724,  0.3499354 ,
         0.54275349],
       [ 0.2513958 , -0.3825604 , -0.24593065, -0.74144624, -0.52271362,
         0.29422769],
       [ 0.99999999,  0.9997763 ,  0.99989617,  0.10992257,  0.11920913,
         0.08510489],
       [-0.99999999,  0.50295697,  0.32210066,  0.99999999,  0.99999999,
         0.03677753],
       [ 0.79080796,  0.99977629,  0.99999999,  0.43606457,  0.66316119,
         0.3033284 ],
       [-0.03541407,  0.25883652, -0.49792355,  0.22743353, -0.39118338,
        -0.54275348],
       [ 0.85161667,  1.0004474 ,  1.        ,  0.20024119,  0.34043987,
         0.23126251],
       [-0.00448109,  0.58658021,  0.47786026,  0.68790636,  0.80661706,
         0.19454313]])

## 7) Regression

In [12]:
from sklearn.svm import SVR

model = SVR()

## 8) One-Class SVM

In [13]:
from sklearn.svm import OneClassSVM

model = OneClassSVM()